# T08: Maintaining a Conversation

A follow-up such as "Who verified them?" only makes sense together with the earlier turns. This notebook compares two ways to pass the conversation to the RAG agent and shows the query rewriting the agent applies on top.

| Mechanism | Idea |
|---|---|
| **Full history** | Send every previous turn verbatim |
| **History summary** | Send an LLM-written summary of the previous turns |
| **Query reformulation** | Rewrite the query into a standalone one before retrieval (always on in `RAG` when there is history) |

## Setup

Run T01 first: it builds the vector store at `backend/db/data_vs.db`. If the store is empty, `load_or_build_vector_store()` builds it from all documents, which takes several minutes.

In [ ]:
import sys

from loguru import logger

from conversational_toolkit.llms.base import LLMMessage, MessageContent, Roles

from ki_kompass_rag.baseline_rag import (
    RETRIEVER_TOP_K,
    SYSTEM_PROMPT,
    ask,
    build_agent,
    build_llm,
    load_or_build_vector_store,
)

logger.remove()
logger.add(sys.stderr, level="WARNING")

vector_store, embedding_model = await load_or_build_vector_store()
llm = build_llm("openai")
agent = build_agent(
    vector_store,
    embedding_model,
    llm,
    top_k=RETRIEVER_TOP_K,
    system_prompt=SYSTEM_PROMPT,
)


def text_message(role: Roles, text: str) -> LLMMessage:
    return LLMMessage(role=role, content=[MessageContent(type="text", text=text)])

## Full history

After each turn, the query and the answer are appended to `history`, which is sent with the next query. Retrieved chunks are not stored; they would make the history grow quickly. Even so, cost and context length grow with every turn.

The third query can only be answered from the history.

In [ ]:
async def run_full_history(queries: list[str]) -> None:
    history: list[LLMMessage] = []
    for query in queries:
        print(f"\n>>> {query}")
        answer = await ask(agent, query, history)
        history += [text_message(Roles.USER, query), answer]


await run_full_history(
    [
        "Which pallets in our portfolio have a third-party verified EPD?",
        "Rewrite what you just said as a pirate, very concisely.",
        "Write minutes of our conversation so far.",
    ]
)

## History summary

Before each turn, an LLM compresses the previous turns into a short summary, which replaces the history. The context stays small, but details from earlier turns can get lost.

In [ ]:
SUMMARY_PROMPT = "Summarize this conversation between a user and an assistant in a few sentences. Do not answer any question."


async def summarize(history: list[LLMMessage]) -> str:
    transcript = "\n".join(f"{m.role}: {m.content[0].text}" for m in history)
    response = await llm.generate(
        [
            text_message(Roles.SYSTEM, SUMMARY_PROMPT),
            text_message(Roles.USER, transcript),
        ]
    )
    return response.content[0].text or ""


async def run_with_summary(queries: list[str]) -> None:
    history: list[LLMMessage] = []
    for query in queries:
        print(f"\n>>> {query}")
        context = []
        if history:
            summary = await summarize(history)
            print(f"Summary: {summary}\n")
            context = [text_message(Roles.SYSTEM, f"Conversation so far: {summary}")]
        answer = await ask(agent, query, context)
        history += [text_message(Roles.USER, query), answer]


await run_with_summary(
    [
        "Which pallets in our portfolio have a third-party verified EPD?",
        "Rewrite what you just said as a pirate, very concisely.",
        "Write minutes of our conversation so far.",
    ]
)

## Query reformulation

"Who verified them?" retrieves nothing useful when embedded on its own. Whenever there is history, `RAG` first asks the LLM to rewrite the query into a standalone one (`make_query_standalone` in `conversational_toolkit.utils.retriever`) and retrieves with the rewrite. This also happened in both runs above.

The DEBUG log of that module shows the original and the reformulated query.

In [ ]:
sink = logger.add(
    sys.stderr,
    level="DEBUG",
    filter="conversational_toolkit.utils.retriever",
    format="{message}",
)
await run_full_history(
    [
        "Which pallets in our portfolio have a third-party verified EPD?",
        "Who verified them?",
    ]
)
logger.remove(sink)